# Pulsar classification — Random Forest
Tune with 5-fold CV on `train.csv`, pick threshold on `val.csv`, report **once** on `test.csv`.

In [ ]:
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (average_precision_score, roc_auc_score, classification_report,
                             confusion_matrix, precision_recall_curve, recall_score,
                             precision_score, f1_score)

# In Colab: upload train.csv, val.csv, test.csv when prompted (the files from split_data.py)
try:
    from google.colab import files
    if not all(os.path.exists(f) for f in ["train.csv", "val.csv", "test.csv"]):
        print("Upload train.csv, val.csv and test.csv")
        files.upload()
except ImportError:
    pass  # running locally

DATA_DIR = "data" if os.path.exists("data/train.csv") else "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
val   = pd.read_csv(f"{DATA_DIR}/val.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

FEATURES = [c for c in train.columns if c != "label"]
X_train, y_train = train[FEATURES].values, train["label"].astype(int).values
X_val,   y_val   = val[FEATURES].values,   val["label"].astype(int).values
X_test,  y_test  = test[FEATURES].values,  test["label"].astype(int).values

SEED = 42
print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("pulsar ratio  train/val/test:", y_train.mean().round(4), y_val.mean().round(4), y_test.mean().round(4))

In [ ]:
def pick_threshold(y_true, scores, min_precision=0.90):
    """Highest-recall threshold that keeps precision >= min_precision.
    Always call this on VALIDATION data, never on test."""
    prec, rec, thr = precision_recall_curve(y_true, scores)
    prec, rec = prec[:-1], rec[:-1]
    ok = prec >= min_precision
    if ok.any():
        i = np.argmax(np.where(ok, rec, -1))
    else:                                   # fallback: best F1
        f1 = 2 * prec * rec / (prec + rec + 1e-12)
        i = np.argmax(f1)
    return float(thr[i])


def report(name, y_true, scores, thr, default_thr):
    """Print metrics + plot confusion matrix and PR curve."""
    out = {}
    for label, t in [("default threshold", default_thr), ("tuned threshold", thr)]:
        pred = (scores >= t).astype(int)
        print(f"\n=== {name} | {label} = {t:.3f} ===")
        print(classification_report(y_true, pred, target_names=["not pulsar", "pulsar"], digits=4))
        cm = confusion_matrix(y_true, pred)
        print("Confusion matrix [[TN FP] [FN TP]]:", cm.tolist())
        out[label] = dict(threshold=t, recall=recall_score(y_true, pred),
                          precision=precision_score(y_true, pred), f1=f1_score(y_true, pred),
                          confusion_matrix=cm.tolist())
    out["pr_auc"] = average_precision_score(y_true, scores)
    out["roc_auc"] = roc_auc_score(y_true, scores)
    print(f"\nPR-AUC: {out['pr_auc']:.4f} | ROC-AUC: {out['roc_auc']:.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    cm = np.array(out["tuned threshold"]["confusion_matrix"])
    ax[0].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax[0].text(j, i, v, ha="center", va="center", fontsize=14)
    ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1])
    ax[0].set_xticklabels(["not pulsar", "pulsar"]); ax[0].set_yticklabels(["not pulsar", "pulsar"])
    ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual")
    ax[0].set_title(f"{name}: confusion matrix (tuned threshold)")
    p, r, _ = precision_recall_curve(y_true, scores)
    ax[1].plot(r, p)
    t = out["tuned threshold"]
    ax[1].scatter([t["recall"]], [t["precision"]], c="red", zorder=3, label="tuned threshold")
    ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
    ax[1].set_title(f"{name}: precision-recall curve (test)"); ax[1].legend()
    plt.tight_layout(); plt.show()
    return out


def save_results(res, fname):
    with open(fname, "w") as f:
        json.dump(res, f, indent=2)
    try:
        from google.colab import files
        files.download(fname)
    except ImportError:
        print("Saved", fname)

## 1. Hyperparameter search (stratified 5-fold CV on train, scored by PR-AUC)
`class_weight="balanced_subsample"` re-weights the minority class inside each tree's bootstrap sample.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint

rf = RandomForestClassifier(class_weight="balanced_subsample", random_state=SEED, n_jobs=-1)
space = {"n_estimators": randint(100, 600),
         "max_depth": randint(4, 25),
         "min_samples_leaf": randint(1, 10),
         "max_features": ["sqrt", "log2", 0.5]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
search = RandomizedSearchCV(rf, space, n_iter=30, cv=cv, scoring="average_precision",
                            random_state=SEED, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
model = search.best_estimator_
print("Best params:", search.best_params_)
print("CV PR-AUC:", round(search.best_score_, 4))

## 2. Choose the decision threshold on validation data

In [ ]:
val_scores = model.predict_proba(X_val)[:, 1]
MIN_PRECISION = 0.90          # lower this to catch more pulsars at the cost of more false alarms
thr = pick_threshold(y_val, val_scores, MIN_PRECISION)
print("Chosen threshold:", round(thr, 3))

## 3. Final evaluation on the test set (look once)

In [ ]:
test_scores = model.predict_proba(X_test)[:, 1]
results = report("Random Forest", y_test, test_scores, thr, default_thr=0.5)
results["best_params"] = {k: (v if isinstance(v, str) else float(v)) for k, v in search.best_params_.items()}
results["cv_pr_auc"] = float(search.best_score_)

## 4. Feature importance (for the write-up)

In [ ]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(figsize=(6, 4), title="Random Forest feature importance"); plt.tight_layout(); plt.show()
results["feature_importance"] = imp.to_dict()
save_results(results, "results_random_forest.json")